# Australian direct emission factors, 2024–25

This notebook implements the generation-weighted aggregation requested in
[electricitymaps-contrib#5383](https://github.com/electricitymaps/electricitymaps-contrib/issues/5383),
following the direct-emissions calculation in the project's
[US methodology](https://github.com/electricitymaps/electricitymaps-contrib/wiki/US-emission-factors).
It calculates fossil-fuel factors for seven Australian grid zones from the
Clean Energy Regulator's designated-generation-facility data.

The results describe **scope 1 emissions per gross kWh generated** during the
Australian financial year **1 July 2024 to 30 June 2025**. They are a reproducible
calculation for review, rather than production configuration overrides.


## Source and units

The [NGER publication](https://cer.gov.au/markets/reports-and-data/nger-reporting-data-and-registers/electricity-sector-emissions-and-generation-data-2024-25)
was published with information as at 27 February 2026. The adjacent
`nger_2024_25.csv` is an unchanged copy of the
[official CSV](https://cer.gov.au/document/greenhouse-and-energy-information-designated-generation-facility-2024-25-0),
retrieved on 29 October 2026. The original encoding is Windows-1252.

For each zone and production mode, we calculate:

`direct gCO2e/kWh = 1000 × sum(scope 1 tonnes CO2e) / sum(gross generation MWh)`

This is equivalent to weighting each facility's unrounded emission factor by its
reported generation. The pre-rounded intensity column and scope 2 emissions are
not used.

NGER's [publication notes](https://cer.gov.au/markets/reports-and-data/nger-reporting-data-and-registers/electricity-sector-emissions-and-generation-data-2023-24)
define generation as including the facility's own electricity use. This produces
lower intensities than a denominator based on electricity sent out. NGER covers
designated facilities whose principal activity is electricity generation;
secondary generation at industrial facilities may be absent.


In [ ]:
from pathlib import Path
import hashlib
import sys

# Work from either the repository root or this notebook's directory.
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "scripts/au_emission_factors.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from scripts.au_emission_factors import calculate_factors, read_nger, write_factors

DATA_DIR = ROOT / "notebooks/au_emission_factors"
source = DATA_DIR / "nger_2024_25.csv"
assert hashlib.sha256(source.read_bytes()).hexdigest() == "cf29fbdbc46a802fe1c925b31aa0c4ad5b747af429c54f259f96745b9a73a648"
rows = read_nger(source)
print(f"Source: {len(rows)} records, {sum(row['Type'] == 'F' for row in rows)} facility reports")


## Selection and zone mapping

- Keep facility records (`Type=F`) connected to the relevant grid. Corporate
  subtotals (`Type=C`) would count the same emissions and production twice.
- Map NEM states to their corresponding `AU-*` zones, with ACT included in NSW.
  Map SWIS to `AU-WA` and Darwin–Katherine (DKIS) to `AU-NT`, consistent with the
  zones' production parsers. Other grids and off-grid stations are excluded.
- Combine black and brown coal into `coal`; natural gas, coal seam methane,
  waste coal mine gas and ethane into `gas`; diesel and kerosene into `oil`.
  NGER identifies a facility's *primary* fuel, so all its generation and reported
  scope 1 emissions are attributed to that mode. Biomass, renewable generation
  and storage require their existing distinct accounting methods.
- Exclude zero-generation records, for which a per-kWh factor is undefined.
  Reject malformed, negative or non-finite numerical values.
- Count the Callide C reports from C S Energy and IG Power Holdings once.
  NGER explicitly documents that they report the same full-year facility data;
  the calculator verifies agreement before dropping the duplicate.
  Preserve both different Moranbah reports: a shared facility name alone does
  not establish duplication. `reporting_records` counts included reports,
  rather than unique physical generating units.


In [ ]:
from dataclasses import asdict

factors = calculate_factors(rows)
results = pd.DataFrame([
    {**asdict(factor), "direct_gco2e_per_kwh": factor.direct_gco2e_per_kwh}
    for factor in factors
])
print(results.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))


## Reproduce the published results

`results.csv` contains the aggregated generation and emissions alongside each
factor. Ratios retain full precision during aggregation and are rounded to two
decimal places only on export. Re-running this cell must reproduce the committed
file byte for byte. The command-line equivalent is in the README.


In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary:
    output = Path(temporary) / "results.csv"
    write_factors(factors, output)
    assert output.read_bytes() == (DATA_DIR / "results.csv").read_bytes()
print(f"Reproduced {len(factors)} zone/mode factors exactly.")


## Interpretation before deployment

These are annual averages of reported designated-facility data, not real-time
unit factors. Small oil-generation samples may represent peaking operation, and
a missing zone/mode combination means that no eligible positive-generation
report was available; it does not imply zero emissions.

Using the results in live Electricity Maps configuration requires reconciling
NGER gross generation with the generation basis of the production parser.
Lifecycle factors additionally require upstream fuel and infrastructure
emissions, as described in the project's EU and US methodologies. Scope 2
emissions from imported station electricity do not supply that upstream term.
Both questions should be resolved before deploying configuration overrides.

## Attribution

Licensed from the Clean Energy Regulator, Commonwealth of Australia under a
[Creative Commons Attribution 4.0 licence](https://creativecommons.org/licenses/by/4.0/).
The CSV source is unchanged. The calculated factors and commentary are derived
material. See the [CER copyright policy](https://cer.gov.au/about-us/our-policies/copyright).
